- 减速 / 刹车（Brake 模式）：

    - 看 前方扇区：前面有车就必须刹车；

    - 同时看 后方窄扇区：

        如果后车离得很近 → 不允许特别猛的刹车（避免人类开车里那种“我刹车你追尾”的危险场景）；

        后面安全 → 允许更激进刹车。

- 加速（Accel 模式）：

    - 看 前方扇区：前面有车就不能急加速；

    - 看 左右扇区：并线/起步时，左右有车就限制加速和转向；

- 巡航（Cruise 模式）：只做轻微保护，主要以策略自身为主。
- 起步的时候需要左右看

In [ ]:
%matplotlib widget


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

# ========= superellipse（前后非对称 + 圆润度对调） =========

def superellipse_points_asym(a, b_front, b_back, p_front, p_back, num=400):
    t = np.linspace(0, 2 * np.pi, num)
    cos_t = np.cos(t)
    sin_t = np.sin(t)

    x = a * np.sign(cos_t) * np.abs(cos_t) ** (2.0 / p_front)

    y = np.empty_like(sin_t)
    mask_front = sin_t >= 0   # 前半部分
    mask_back  = ~mask_front  # 后半部分

    y[mask_front] =  b_front * (np.abs(sin_t[mask_front]) ** (2.0 / p_front))
    y[mask_back]  = -b_back  * (np.abs(sin_t[mask_back])  ** (2.0 / p_back))

    return x, y


def smooth_step(x, sharpness=1.0):
    return 1.0 / (1.0 + np.exp(-sharpness * x))


# ========= Safe Bubble 参数（含起步左右观察） =========

def safe_bubble_params(v_kmh, a_cmd, p_base, anisotropy):
    v = v_kmh / 3.6
    V_HIGH = 100.0
    s = np.clip(v_kmh / V_HIGH, 0.0, 1.0)

    # === 起步阶段 0~10 km/h：强调左右观察 ===
    v_start = 10.0
    w_start = np.clip(1.0 - v_kmh / v_start, 0.0, 1.0)

    # === 基础尺寸 ===
    b_base = 0.2 + 0.05 * v_kmh          # 前后基准尺度
    a = b_base * (0.5 + 0.5 * (1.0 - s)) # 速度越高横向相对收窄
    a = a / anisotropy

    # 起步时左右再加宽（最多 +60%）
    a = a * (1.0 + 0.60 * w_start)

    # === 前长后短 + 起步稍微缩短前方 ===
    front_bias = 0.4
    b_front = b_base * (1.0 + front_bias) * (1.0 - 0.3 * w_start)
    b_back  = b_base * (1.0 - front_bias)

    # === 曲率对调 ===
    delta   = 0.8
    p_front = np.clip(p_base + delta, 1.0, 5.0)  # 前面更圆润
    p_back  = np.clip(p_base - delta, 1.0, 5.0)  # 后面更尖一点

    # === 加减速导致泡泡前后偏移 ===
    shift = np.tanh(a_cmd / 2.0)                 # -1 ~ 1
    y_shift = shift * 0.3 * (b_front + b_back) * 0.5

    # 行为模式显示用
    w_accel  = smooth_step(a_cmd,  0.7)
    w_brake  = smooth_step(-a_cmd, 0.7)
    w_cruise = 1.0 - max(w_accel, w_brake)

    return {
        "a": a,
        "b_front": b_front,
        "b_back":  b_back,
        "p_front": p_front,
        "p_back":  p_back,
        "y_shift": y_shift,
        "w_accel": w_accel,
        "w_brake": w_brake,
        "w_cruise": w_cruise,
        "w_start": w_start,
    }


# ========= 绘图部分 =========

fig, ax = plt.subplots(figsize=(6, 6))
plt.subplots_adjust(bottom=0.25)  # 只剩 3 个滑条，底边可以高一点

def draw_bubble(v_kmh, a_cmd, p_base, anisotropy):
    ax.clear()
    params = safe_bubble_params(v_kmh, a_cmd, p_base, anisotropy)

    x, y = superellipse_points_asym(
        params["a"],
        params["b_front"],
        params["b_back"],
        params["p_front"],
        params["p_back"]
    )
    y = y + params["y_shift"]

    ax.fill(x, y, alpha=0.25, label="Safe bubble")

    # Ego 车
    ax.scatter([0], [0], s=50, c="purple")
    ax.text(0, 0, " Ego", ha="left", va="bottom")

    rmax = max(params["a"], params["b_front"], params["b_back"]) + 5.0
    ax.set_xlim(-rmax, rmax)
    ax.set_ylim(-rmax, rmax)

    ax.set_aspect("equal", "box")
    ax.set_xlabel("x (m, right)")
    ax.set_ylabel("y (m, forward)")

    ax.set_title(
        f"Safe Bubble（Acc ver）\n"
        f"v = {v_kmh:.1f} km/h, a ≈ {a_cmd:.2f} m/s²"
    )

    info = (
        f"w_start  = {params['w_start']:.2f}\n"
        f"w_accel  = {params['w_accel']:.2f}\n"
        f"w_brake  = {params['w_brake']:.2f}"
    )
    ax.text(
        0.02, 0.98, info,
        transform=ax.transAxes,
        va="top", ha="left",
        fontsize=9,
        bbox=dict(boxstyle="round", alpha=0.25),
    )

    ax.legend(loc="upper right")
    ax.grid(True)


# ========= 滑条设置 =========

# 初始值
v0   = 0.0   # 当前车速（km/h）
p0   = 2.0   # 基础曲率
an0  = 1.0   # 各向同性
dt   = 1.0   # 虚拟时间步长，用来算加速度

# 记住“上一帧速度”，用来算 a_cmd
last_speed = v0
last_accel = 0.0  # 仅用于显示

ax_speed = plt.axes([0.15, 0.18, 0.7, 0.03])
ax_p     = plt.axes([0.15, 0.13, 0.7, 0.03])
ax_aniso = plt.axes([0.15, 0.08, 0.7, 0.03])

s_speed = Slider(ax_speed, "Speed (km/h)", 0.0, 120.0, valinit=v0,  valstep=1.0)
s_p     = Slider(ax_p,     "Base p",       1.0, 4.0,   valinit=p0,  valstep=0.1)
s_aniso = Slider(ax_aniso, "Anisotropy",   0.5, 2.0,   valinit=an0, valstep=0.05)

def update(_):
    global last_speed, last_accel

    v_now = s_speed.val
    dv = v_now - last_speed
    a_cmd = dv / dt    # 简单差分当作加速度

    last_speed = v_now
    last_accel = a_cmd

    draw_bubble(
        v_now,
        a_cmd,
        s_p.val,
        s_aniso.val,
    )
    fig.canvas.draw_idle()

s_speed.on_changed(update)
s_p.on_changed(update)
s_aniso.on_changed(update)

# 初次绘制（静止，a=0）
draw_bubble(v0, 0.0, p0, an0)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

# ========= superellipse（前后非对称 + 圆润度对调） =========

def superellipse_points_asym(a, b_front, b_back, p_front, p_back, num=400):
    t = np.linspace(0, 2 * np.pi, num)
    cos_t = np.cos(t)
    sin_t = np.sin(t)

    x = a * np.sign(cos_t) * np.abs(cos_t) ** (2.0 / p_front)

    y = np.empty_like(sin_t)
    mask_front = sin_t >= 0   # 前半部分
    mask_back  = ~mask_front  # 后半部分

    y[mask_front] =  b_front * (np.abs(sin_t[mask_front]) ** (2.0 / p_front))
    y[mask_back]  = -b_back  * (np.abs(sin_t[mask_back])  ** (2.0 / p_back))

    return x, y


def smooth_step(x, sharpness=1.0):
    return 1.0 / (1.0 + np.exp(-sharpness * x))


# ========= Safe Bubble 参数计算 =========

def safe_bubble_params(v_kmh, a_cmd, p_base, anisotropy):
    """
    v_kmh      : 车速 (km/h)
    a_cmd      : 纵向加速度 (m/s^2)，>0 加速，<0 刹车
    p_base     : 超椭圆基础曲率
    anisotropy : 横向缩放因子（来自滑条）

    返回:
      - a        : 左右半径（由“侧向注意力 + 车道宽度”决定）
      - b_front  : 前方半径（由停止距离 + 前向注意力决定）
      - b_back   : 后方半径（由后向注意力 + 刹车情况决定）
      - p_front  : 前方圆润度
      - p_back   : 后方圆润度
      - y_shift  : 泡泡整体在 y 方向的偏移（加减速）
      - w_accel / w_brake / w_cruise / w_start : 仅用于显示
    """
    # ---------- 基本量 ----------
    v = v_kmh / 3.6                        # m/s
    v_norm = np.clip(v_kmh / 120.0, 0.0, 1.0)  # 0 ~ 1, 120km/h 归一化到 1

    # 起步阶段（0~10 km/h）权重，用来强调左右观察
    v_start = 10.0
    w_start = np.clip(1.0 - v_kmh / v_start, 0.0, 1.0)

    # 加速度归一化（人类感觉“踩油”/“点刹”）
    a_norm = np.tanh(a_cmd / 2.0)   # -1 ~ 1

    # ---------- 注视权重（注意力分布） ----------
    # 直觉：速度越高 → 前方比重增加；刹车时 → 有一部分注意力分给后方；
    # 起步时 → 左右比重再拉高一点
    w_front_raw = 0.4 + 0.4 * v_norm + 0.2 * max(a_norm, 0.0)     # 加速时更看前方
    w_back_raw  = 0.2 + 0.3 * max(-a_norm, 0.0)                   # 刹车时更看后方
    w_side_raw  = 1.0 - (w_front_raw + w_back_raw)

    # 如果 w_side_raw 为负，说明前+后已经把“预算”占满了，就把前/后按比例缩一下
    if w_side_raw < 0.0:
        scale = 1.0 / (w_front_raw + w_back_raw + 1e-6)
        w_front_raw *= scale
        w_back_raw  *= scale
        w_side_raw   = 0.0

    # 起步时左右再提升一截
    w_side_raw += 0.5 * w_start

    # 重新归一化成概率分布
    w_sum = w_front_raw + w_side_raw + w_back_raw + 1e-6
    w_front = w_front_raw / w_sum
    w_side  = w_side_raw  / w_sum
    w_back  = w_back_raw  / w_sum

    # ---------- 前向：按停止距离来定一个“基础前向尺度” ----------
    a_comf = 4.0         # 认为最大舒适减速度 4 m/s^2
    t_react = 1.0        # 反应时间 1s
    b_front_base = 8.0 + 0.15 * v_kmh    # 线性尺度
    b_front_base = np.clip(b_front_base, 8.0, 55.0)

    b_front = b_front_base * (0.8 + 0.4 * w_front) * (1.0 - 0.3 * w_start)

    # ---------- 后向：基础一个小范围，刹车/后向注意力高时放大 ----------
    b_back_base = 3.0 + 0.2 * b_front_base      # 永远不会只有几米那么小
    b_back = b_back_base * (0.7 + 0.6 * w_back)

    # ---------- 侧向：基于车宽 + 侧向注意力 ----------
    lane_width = 3.6          # 一条车道宽度 (m)
    car_width  = 1.9          # 假设乘用车宽度 (m)
    car_half   = car_width / 2.0

    # 基础侧向安全裕度：车身外再留 ~0.6m
    base_margin = 0.6         # 单侧 (m)

    # 速度归一化，用来让高速时稍微多一点横向安全
    v_norm = np.clip(v_kmh / 120.0, 0.0, 1.0)

    # 侧向半径 = 车半宽 + 固定安全裕度 + 少量随速度/侧向注意力变化的额外裕度
    extra_margin = (0.3 + 0.7 * w_side) * v_norm * car_half  # 最高再加到约 1 * car_half

    side_radius = car_half + base_margin + extra_margin

    # 限制在 [车宽 + 一点点, 单侧不超过一个车道]
    side_radius = np.clip(
        side_radius,
        car_half + 0.4,   # 最小：车身外至少再留 40cm
        lane_width       # 最大：单侧不超过一条车道宽
    )

    # 加各向异性因子
    a = side_radius / anisotropy


    # 加 anisotropy：你可以用滑条把横向再缩放/放大一点
    a = side_radius / anisotropy

    # ---------- 曲率：前圆 / 后尖 ----------
    delta = 0.8
    p_front = np.clip(p_base + delta, 1.0, 5.0)
    p_back  = np.clip(p_base - delta, 1.0, 5.0)

    # ---------- 加减速导致泡泡整体前后偏移 ----------
    shift = np.tanh(a_cmd / 2.0)            # -1 ~ 1
    y_shift = shift * 0.3 * (b_front + b_back) * 0.5

    # 显示用行为权重
    w_accel  = smooth_step(a_cmd, 0.7)
    w_brake  = smooth_step(-a_cmd, 0.7)
    w_cruise = 1.0 - max(w_accel, w_brake)

    return {
        "a": a,
        "b_front": b_front,
        "b_back":  b_back,
        "p_front": p_front,
        "p_back":  p_back,
        "y_shift": y_shift,
        "w_accel": w_accel,
        "w_brake": w_brake,
        "w_cruise": w_cruise,
        "w_start": w_start,
        "w_front": w_front,
        "w_side":  w_side,
        "w_back":  w_back,
    }

# ========= 绘图 =========

fig, ax = plt.subplots(figsize=(6, 6))
plt.subplots_adjust(bottom=0.3)

def draw_bubble(v_kmh, a_cmd, p_base, anisotropy):
    ax.clear()
    params = safe_bubble_params(v_kmh, a_cmd, p_base, anisotropy)

    x, y = superellipse_points_asym(
        params["a"],
        params["b_front"],
        params["b_back"],
        params["p_front"],
        params["p_back"]
    )
    y += params["y_shift"]

    ax.fill(x, y, alpha=0.25, label="Safe bubble")

    ax.scatter([0], [0], s=50, c="purple")
    ax.text(0, 0, " Ego", ha="left", va="bottom")

    rmax = max(params["a"], params["b_front"], params["b_back"]) + 5
    ax.set_xlim(-rmax, rmax)
    ax.set_ylim(-rmax, rmax)

    ax.set_aspect("equal")
    ax.set_xlabel("x (m, right)")
    ax.set_ylabel("y (m, forward)")

    ax.set_title(
        f"Safe Bubble (Gaze-inspired)\n"
        f"v={v_kmh:.1f} km/h, a={a_cmd:.2f}"
    )

    txt = (
        f"w_front = {params['w_front']:.2f}\n"
        f"w_side  = {params['w_side']:.2f}\n"
        f"w_back  = {params['w_back']:.2f}\n"
        f"w_start = {params['w_start']:.2f}"
    )
    ax.text(0.02, 0.98, txt, transform=ax.transAxes,
            va="top", ha="left", bbox=dict(boxstyle="round", alpha=0.25), fontsize=9)

    ax.grid(True)
    ax.legend(loc="upper right")


# ========= 滑条 =========

v0, a0, p0, an0 = 0.0, 0.0, 2.0, 1.0

ax_speed = plt.axes([0.15, 0.22, 0.7, 0.03])
ax_accel = plt.axes([0.15, 0.17, 0.7, 0.03])
ax_p     = plt.axes([0.15, 0.12, 0.7, 0.03])
ax_aniso = plt.axes([0.15, 0.07, 0.7, 0.03])

s_speed = Slider(ax_speed, "Speed (km/h)", 0, 120, valinit=v0, valstep=1)
s_accel = Slider(ax_accel, "Accel (m/s²)", -4, 4, valinit=a0, valstep=0.1)
s_p     = Slider(ax_p,     "Base p", 1, 4, valinit=p0, valstep=0.1)
s_aniso = Slider(ax_aniso, "Anisotropy", 0.5, 2.0, valinit=an0, valstep=0.05)

def update(_):
    draw_bubble(s_speed.val, s_accel.val, s_p.val, s_aniso.val)
    fig.canvas.draw_idle()

for slider in [s_speed, s_accel, s_p, s_aniso]:
    slider.on_changed(update)

draw_bubble(v0, a0, p0, an0)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

# ========= 工具函数 =========

def superellipse_points_asym(a, b_front, b_back, p_front, p_back, num=400):
    """
    非对称超级椭圆：
      上半部分(y>=0) 用 (b_front, p_front)
      下半部分(y< 0) 用 (b_back,  p_back)
      |x/a|^p_* + |y/b_*|^p_* = 1
    """
    t = np.linspace(0, 2 * np.pi, num)
    cos_t = np.cos(t)
    sin_t = np.sin(t)

    # x 用前半部分的 p_front（和 JAX 逻辑保持简单一致）
    x = a * np.sign(cos_t) * np.abs(cos_t) ** (2.0 / p_front)

    y = np.empty_like(sin_t)
    mask_front = sin_t >= 0   # 前半部分
    mask_back  = ~mask_front  # 后半部分

    y[mask_front] =  b_front * (np.abs(sin_t[mask_front]) ** (2.0 / p_front))
    y[mask_back]  = -b_back  * (np.abs(sin_t[mask_back])  ** (2.0 / p_back))

    return x, y


# ========= Safe Bubble 参数计算（和 JAX 版本对齐 + y_shift） =========

def safe_bubble_params_np(v_kmh, a_cmd, p_base, anisotropy):
    """
    基本几何和 JAX 版 _compute_safe_bubble_reward 对齐：
      - w_front / w_side / w_back / w_start
      - b_front / b_back
      - side_radius: base_margin=0.2,
                     extra_margin=(0.15+0.7*w_side)*v_norm*car_half
                     clip 到 [car_half+0.15, lane_width]
      - a_lat = side_radius / anisotropy

    额外多一个 y_shift 只用于可视化（reward 里 a_cmd=0 → y_shift≈0）。
    """
    v_kmh = float(v_kmh)
    a_cmd = float(a_cmd)

    # ---------- 基本量 ----------
    v = v_kmh / 3.6                         # m/s
    v_norm = np.clip(v_kmh / 120.0, 0.0, 1.0)

    # 起步阶段（0~10 km/h）权重，用来强调左右观察
    v_start = 10.0
    w_start = np.clip(1.0 - v_kmh / v_start, 0.0, 1.0)

    # 加速度归一化（这里可视化版用滑条 a_cmd；JAX 里固定 0）
    a_norm = np.tanh(a_cmd / 2.0)   # -1 ~ 1

    # ---------- 注视权重（注意力分布） ----------
    w_front_raw = 0.4 + 0.4 * v_norm + 0.2 * max(a_norm, 0.0)
    w_back_raw  = 0.2 + 0.3 * max(-a_norm, 0.0)
    w_side_raw  = 1.0 - (w_front_raw + w_back_raw)

    # 保证侧向权重非负
    w_side_raw = max(w_side_raw, 0.0)

    # 起步时左右再提升一截
    w_side_raw = w_side_raw + 0.5 * w_start

    # 归一化成概率分布
    w_sum = w_front_raw + w_side_raw + w_back_raw + 1e-6
    w_front = w_front_raw / w_sum
    w_side  = w_side_raw  / w_sum
    w_back  = w_back_raw  / w_sum

    # ========= ☆ 关键修改：用“反应距离 + 时间车距”算前向基础距离 =========
    # 静态缓冲距离（完全停住时，和前车保持一点点距离）
    d_static = 1.5   # m，大概 1/3 辆车

    # 城市工况我们希望车距偏小一点：
    #   0 km/h  → t_gap ≈ 0.4 s（几乎只有静态距离）
    #   80 km/h → t_gap ≈ 0.75 s → 大约 18m
    v_norm_city = np.clip(v_kmh / 80.0, 0.0, 1.0)
    t_gap_min = 0.4
    t_gap_max = 0.75
    t_gap = t_gap_min + (t_gap_max - t_gap_min) * v_norm_city

    # ===== TTC-aligned longitudinal sizing =====
    ttc_threshold_s = 1.5      # 对齐 TimeToCollisionMetric 的阈值
    a_brake = 4.0              # m/s^2（舒适偏强；想更激进就调到 6~8）
    T_react = ttc_threshold_s  # 直接用 TTC 作为反应窗口（最直观对齐）

    b_front_base = d_static + v * T_react + (v**2) / (2.0 * a_brake)
    b_front_base = np.clip(b_front_base, 1.5, 60.0)  # 上限可放宽点（高速会更大）


    # 再叠加你原来的“前向注意力”和“起步时系数”
    b_front = b_front_base * (0.8 + 0.4 * w_front) * (1.0 - 0.3 * w_start)

    T_rear = 1.0
    a_rear = 4.0
    d_static_rear = 1.0

    b_back_base = d_static_rear + v * T_rear + (v**2) / (2.0 * a_rear)
    # 让后向比前向短一些（可选）
    b_back_base = np.minimum(b_back_base, 0.6 * b_front_base)
    b_back_base = np.clip(b_back_base, 1.0, 40.0)

    b_back = b_back_base * (0.25 + 0.8 * w_back)


    # ---------- 侧向：基于车宽 + 侧向注意力（严格用你现在的参数） ----------
    lane_width = 3.6          # 一条车道 (m)
    car_width  = 1.9          # 假设轿车宽度 (m)
    car_half   = car_width / 2.0

    base_margin = 0.2         # 车身外再留约 0.2m
    v_norm2 = np.clip(v_kmh / 120.0, 0.0, 1.0)

    extra_margin = (0.15 + 0.7 * w_side) * v_norm2 * car_half
    side_radius = car_half + base_margin + extra_margin

    # 单侧至少车宽+0.15m，上限不超过一条车道宽
    side_radius = np.clip(
        side_radius,
        car_half + 0.15,
        lane_width,
    )

    # 各向异性缩放
    a_lat = side_radius / max(anisotropy, 1e-3)

    # ---------- 曲率：前圆 / 后尖 ----------
    delta = 0.8
    p_front = np.clip(p_base + delta, 1.0, 5.0)
    p_back  = np.clip(p_base - delta, 1.0, 5.0)

    # ---------- 加减速导致泡泡整体前后偏移（只用于可视化） ----------
    # a_cmd > 0 加速 → shift > 0 → 泡泡往前
    # a_cmd < 0 减速 → shift < 0 → 泡泡往后
    shift = np.tanh(a_cmd / 2.0)            # -1 ~ 1
    y_shift = shift * 0.25 * (b_front + b_back) * 0.3

    return {
        "a": a_lat,
        "b_front": b_front,
        "b_back":  b_back,
        "p_front": p_front,
        "p_back":  p_back,
        "y_shift": y_shift,
        "w_start": w_start,
        "w_front": w_front,
        "w_side":  w_side,
        "w_back":  w_back,
    }


# ========= 绘图 =========

fig, ax = plt.subplots(figsize=(6, 6))
plt.subplots_adjust(bottom=0.3)

def draw_bubble(v_kmh, a_cmd, p_base, anisotropy):
    ax.clear()
    params = safe_bubble_params_np(v_kmh, a_cmd, p_base, anisotropy)

    x, y = superellipse_points_asym(
        params["a"],
        params["b_front"],
        params["b_back"],
        params["p_front"],
        params["p_back"]
    )
    # ⭐ 关键：整体前后平移（可视化效果）
    y = y + params["y_shift"]

    ax.fill(x, y, alpha=0.25, label="Safe bubble")

    # Ego 车
    ax.scatter([0], [0], s=50, c="purple")
    ax.text(0, 0, " Ego", ha="left", va="bottom")

    rmax = max(params["a"], params["b_front"], params["b_back"]) + 2
    ax.set_xlim(-rmax, rmax)
    ax.set_ylim(-rmax, rmax)

    ax.set_aspect("equal")
    ax.set_xlabel("x (m, right)")
    ax.set_ylabel("y (m, forward)")

    ax.set_title(
        f"Safe Bubble (Gaze-inspired, geometry + shift)\n"
        f"v={v_kmh:.1f} km/h, a={a_cmd:.2f}, p_base={p_base:.2f}"
    )

    txt = (
        f"w_front = {params['w_front']:.2f}\n"
        f"w_side  = {params['w_side']:.2f}\n"
        f"w_back  = {params['w_back']:.2f}\n"
        f"w_start = {params['w_start']:.2f}\n"
        f"a      = {params['a']:.2f} m\n"
        f"b_front= {params['b_front']:.2f} m\n"
        f"b_back = {params['b_back']:.2f} m\n"
        f"y_shift= {params['y_shift']:.2f} m"
    )
    ax.text(
        0.02, 0.98, txt,
        transform=ax.transAxes,
        va="top", ha="left",
        bbox=dict(boxstyle="round", alpha=0.25),
        fontsize=9,
    )

    ax.grid(True)
    ax.legend(loc="upper right")


# ========= 滑条 =========

v0, a0, p0, an0 = 0.0, 0.0, 2.0, 1.0

ax_speed = plt.axes([0.15, 0.22, 0.7, 0.03])
ax_accel = plt.axes([0.15, 0.17, 0.7, 0.03])
ax_p     = plt.axes([0.15, 0.12, 0.7, 0.03])
ax_aniso = plt.axes([0.15, 0.07, 0.7, 0.03])

s_speed = Slider(ax_speed, "Speed (km/h)", 0, 120, valinit=v0, valstep=1)
s_accel = Slider(ax_accel, "Accel (m/s²)", -4, 4, valinit=a0, valstep=0.1)
s_p     = Slider(ax_p,     "Base p", 1, 4, valinit=p0, valstep=0.1)
s_aniso = Slider(ax_aniso, "Anisotropy", 0.5, 2.0, valinit=an0, valstep=0.05)

def update(_):
    draw_bubble(s_speed.val, s_accel.val, s_p.val, s_aniso.val)
    fig.canvas.draw_idle()

for slider in [s_speed, s_accel, s_p, s_aniso]:
    slider.on_changed(update)

draw_bubble(v0, a0, p0, an0)
plt.show()


In [ ]:
import jax.numpy as jnp

# 你保存的文件：vmax/simulator/metrics/safe_bubble_soft.py
from vmax.simulator.metrics import safe_bubble as sb


def print_bubble_table_from_safe_bubble_soft(
    speeds_kmh=(0, 5, 10, 20, 30, 50, 80, 100, 120),
    accels_mps2=(-4, -2, 0, 2, 4),
    ego_width_m=1.9,  # 没有 state 时，用一个固定宽度（和你之前假设一致）
    # --- bubble longitudinal (table-aligned) ---
    ttc_front=1.5,
    ttc_rear=1.0,
    d_static=1.5,
    d_static_rear=1.0,
    clip_front=80.0,
    clip_rear=40.0,
    rear_frac=0.6,
    gaze_gain=0.20,
    # --- bubble shape ---
    anisotropy=1.0,
    p_base=2.0,
    delta_p=0.8,
):
    print("=" * 130)
    print("Safe-bubble extents (CALLING safe_bubble_soft.py internals)")
    print("Columns: v_kmh, a_cmd, a_lat, b_front, b_back, front_base, rear_base, w_front, w_back, w_start, scaleF")
    print("=" * 130)

    ego_width = jnp.array(ego_width_m, dtype=jnp.float32)

    for v_kmh in speeds_kmh:
        for a_cmd in accels_mps2:
            v_kmh_j = jnp.array(float(v_kmh), dtype=jnp.float32)
            a_cmd_j = jnp.array(float(a_cmd), dtype=jnp.float32)
            ego_speed = jnp.array(float(v_kmh) / 3.6, dtype=jnp.float32)

            # 1) 直接调用你要求对齐的 gaze 权重（safe_bubble_soft.py 里同一份逻辑）
            w_front, w_side, w_back, w_start = sb._gaze_weights(v_kmh_j, a_cmd_j)

            # 2) 直接调用 bubble 轴长计算（safe_bubble_soft.py 里同一份逻辑）
            a_lat, b_front, b_back, p_front, p_back = sb._bubble_axes_from_state(
                ego_speed_mps=ego_speed,
                ego_width=ego_width,
                v_kmh=v_kmh_j,
                a_cmd=a_cmd_j,
                ttc_front=ttc_front,
                ttc_rear=ttc_rear,
                d_static=d_static,
                d_static_rear=d_static_rear,
                clip_front=clip_front,
                clip_rear=clip_rear,
                rear_frac=rear_frac,
                gaze_gain=gaze_gain,
                anisotropy=anisotropy,
                p_base=p_base,
                delta_p=delta_p,
            )

            # 3) 同时把 front_base / rear_base / scaleF 也按同一套公式算出来打印（便于核对）
            front_base = d_static + ego_speed * float(ttc_front)
            front_base = jnp.clip(front_base, d_static, float(clip_front))

            rear_base = d_static_rear + ego_speed * float(ttc_rear)
            rear_base = jnp.minimum(rear_base, float(rear_frac) * front_base)
            rear_base = jnp.clip(rear_base, d_static_rear, float(clip_rear))

            scaleF = 1.0 + float(gaze_gain) * (w_front - 0.5)

            # ---- penalty probe points on front centerline (x=0) ----
            min_score = 0.20
            penetration_max = 0.50  # 障碍物 深入的尺度 仿真返回的
            k = -jnp.log(min_score) / jnp.maximum(penetration_max, 1e-6)

            def score_from_pnorm(pnorm):
                pen = jnp.maximum(1.0 - pnorm, 0.0)   # inside depth
                score_in = jnp.exp(-k * pen)
                score_in = jnp.clip(score_in, min_score, 1.0)
                return jnp.where(pnorm > 1.0, 1.0, score_in)

            # probe at y = alpha * b_front, x=0  => pnorm = alpha
            s_out = float(score_from_pnorm(jnp.array(1.20, jnp.float32)))  # outside
            s_095 = float(score_from_pnorm(jnp.array(0.95, jnp.float32)))  # slightly inside
            s_070 = float(score_from_pnorm(jnp.array(0.70, jnp.float32)))  # deeper
            s_040 = float(score_from_pnorm(jnp.array(0.40, jnp.float32)))  # very deep

                        
            print(
                f"v={v_kmh:>3} km/h  a={a_cmd:>4}  "
                f"a_lat={float(a_lat):>5.2f}  "
                f"b_front={float(b_front):>6.2f}  b_back={float(b_back):>6.2f}  "
                f"front_base={float(front_base):>6.2f}  rear_base={float(rear_base):>6.2f}  "
                f"wF={float(w_front):>4.2f}  wB={float(w_back):>4.2f}  wS={float(w_start):>4.2f}  "
                f"sF={float(scaleF):>4.2f}"
                f"score(out/0.95/0.70/0.40)={s_out:>4.2f}/{s_095:>4.2f}/{s_070:>4.2f}/{s_040:>4.2f}"
            )


# 直接跑
print_bubble_table_from_safe_bubble_soft()


In [4]:
import importlib
import vmax.simulator.metrics.safe_bubble as sb
importlib.reload(sb)


<module 'vmax.simulator.metrics.safe_bubble' from '/home/rtgtx7/Desktop/AD/v-max/vmax/simulator/metrics/safe_bubble.py'>

In [1]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display




from vmax.simulator.metrics.safe_bubble import bubble_params, bubble_score_from_xy

def render_heatmap(
    speed_kmh=0.0,
    accel_mps2=0.0,
    ego_width_m=1.9,

    # geometry knobs
    ttc_front=3.0,
    ttc_rear=1.5,
    d_static=1.5,
    d_static_rear=1.0,
    rear_frac=0.6,
    clip_front=100.0,
    clip_rear=40.0,

    side_margin_min=0.15,
    side_margin_max=0.45,
    anisotropy=1.0,

    p_base=2.5,
    delta_p=0.5,

    # penalty knobs
    min_score=0.20,
    penetration_max_m=8.0,

    # gating knobs (只做显示层 gating)
    front_on_kmh=0.5,
    side_on_kmh=0.5,
    rear_always_on=True,

    # grid
    xlim=6.0,
    y_back=40.0,
    y_front=100.0,
    nx=241,
    ny=241,
):
    # ---- 1) call safe_bubble.py geometry ----
    a_lat, bF, bB, pF, pB = bubble_params(
        speed_kmh=jnp.array(speed_kmh, jnp.float32),
        accel_mps2=jnp.array(accel_mps2, jnp.float32),
        ego_width_m=jnp.array(ego_width_m, jnp.float32),

        ttc_front=float(ttc_front),
        ttc_rear=float(ttc_rear),
        d_static=float(d_static),
        d_static_rear=float(d_static_rear),
        clip_front=float(clip_front),
        clip_rear=float(clip_rear),
        rear_frac=float(rear_frac),

        side_margin_min=float(side_margin_min),
        side_margin_max=float(side_margin_max),
        anisotropy=float(anisotropy),

        p_base=float(p_base),
        delta_p=float(delta_p),
    )

    # ---- 2) build grid ----
    xs = np.linspace(-xlim, xlim, nx, dtype=np.float32)
    ys = np.linspace(-y_back, y_front, ny, dtype=np.float32)
    X, Y = np.meshgrid(xs, ys)
    x = jnp.asarray(X, jnp.float32)
    y = jnp.asarray(Y, jnp.float32)

    # ---- 3) gating (split front-side vs rear-side) ----
    v = jnp.array(speed_kmh, jnp.float32)
    parked = v <= jnp.array(0.5, jnp.float32)

    front_on = (v > jnp.array(front_on_kmh, jnp.float32)) & (~parked)

    if rear_always_on:
        rear_on = jnp.array(True)
    else:
        rear_on = v <= jnp.array(5.0, jnp.float32)

    # side gate split:
    # - front side: parked 时关
    # - rear  side: parked 时开（你要的“停车也看后方几米区域”）
    side_on_front = (v > jnp.array(side_on_kmh, jnp.float32)) & (~parked)
    side_on_rear  = (v > jnp.array(side_on_kmh, jnp.float32)) | parked

    # per-point lateral allow (N,T grid)
    allow_lat = jnp.where(y >= 0.0, side_on_front, side_on_rear)

    # IMPORTANT: do NOT blow up a_lat; keep scale stable and drop x contribution when lat is off
    x_eff = jnp.where(allow_lat, x, jnp.zeros_like(x))
    a_lat_eff = a_lat  # keep real a_lat

    # ---- 4) call safe_bubble.py score ----
    score_raw = bubble_score_from_xy(
        x=x_eff,
        y=y,
        a_lat=a_lat_eff,
        b_front=bF,
        b_back=bB,
        p_front=pF,
        p_back=pB,
        min_score=float(min_score),
        penetration_max_m=float(penetration_max_m),
    )

    # ---- 5) direction gating ----
    score = jnp.where((y >= 0.0) & (~front_on), 1.0, score_raw)
    score = jnp.where((y <  0.0) & (~rear_on),  1.0, score)

    S = np.asarray(score)

    # ---- 6) plot ----
    plt.figure(figsize=(8, 6))
    plt.imshow(
        S,
        origin="lower",
        aspect="auto",
        extent=[xs[0], xs[-1], ys[0], ys[-1]],
        vmin=float(min_score),
        vmax=1.0,
    )
    plt.colorbar(label="safe-bubble score (1=no penalty, lower=more penalty)")

    levels = sorted(set([float(min_score), 0.3, 0.5, 0.7, 0.85, 0.95, 0.99]))
    levels = [lv for lv in levels if float(min_score) <= lv <= 1.0]
    if len(levels) >= 2:
        plt.contour(xs, ys, S, levels=levels, linewidths=0.8)

    plt.axhline(0.0, linewidth=0.8)
    plt.axvline(0.0, linewidth=0.8)

    plt.xlabel("x (m) lateral")
    plt.ylabel("y (m) longitudinal (front +)")
    plt.title(
    f"safe-bubble score heatmap | v={speed_kmh:.1f} km/h, a={accel_mps2:.1f} m/s²\n"
    f"a_lat={float(a_lat):.2f}, b_front={float(bF):.2f}, b_back={float(bB):.2f}, "
    f"pF={float(pF):.2f}, pB={float(pB):.2f} | "
    f"gate(front={bool(front_on)}, rear={bool(rear_on)}, "
    f"side_front={bool(side_on_front)}, side_rear={bool(side_on_rear)})")

    plt.show()


# ---------- sliders ----------
w_speed = widgets.FloatSlider(description="speed(km/h)", min=0, max=120, step=1, value=0)
w_accel = widgets.FloatSlider(description="accel(m/s²)", min=-4, max=4, step=0.5, value=0)

w_tfc = widgets.FloatSlider(description="ttc_front", min=0.5, max=4.0, step=0.1, value=3.0)
w_trc = widgets.FloatSlider(description="ttc_rear",  min=0.5, max=4.0, step=0.1, value=1.5)
w_dsf = widgets.FloatSlider(description="d_static", min=0.0, max=5.0, step=0.1, value=1.5)
w_dsr = widgets.FloatSlider(description="d_static_rear", min=0.0, max=5.0, step=0.1, value=1.0)
w_rearfrac = widgets.FloatSlider(description="rear_frac", min=0.1, max=1.0, step=0.05, value=0.6)

w_sm_min = widgets.FloatSlider(description="side_min", min=0.0, max=0.8, step=0.02, value=0.15)
w_sm_max = widgets.FloatSlider(description="side_max", min=0.0, max=1.5, step=0.05, value=0.45)
w_aniso  = widgets.FloatSlider(description="anisotropy", min=0.5, max=3.0, step=0.1, value=1.0)

w_pbase = widgets.FloatSlider(description="p_base", min=1.0, max=6.0, step=0.1, value=2.5)
w_dp    = widgets.FloatSlider(description="delta_p", min=0.0, max=3.0, step=0.1, value=0.5)

w_mins   = widgets.FloatSlider(description="min_score", min=0.01, max=0.8, step=0.01, value=0.20)
w_penmax = widgets.FloatSlider(description="pen_max(m)", min=0.2, max=8.0, step=0.1, value=8.0)

w_front_on = widgets.FloatSlider(description="front_on_kmh", min=0, max=10, step=0.1, value=0.5)
w_side_on  = widgets.FloatSlider(description="side_on_kmh",  min=0, max=10, step=0.1, value=0.5)
w_rear_always = widgets.Checkbox(description="rear_always_on", value=True)

w_xlim   = widgets.FloatSlider(description="xlim(m)", min=2, max=12, step=0.5, value=6.0)
w_yback  = widgets.FloatSlider(description="y_back(m)", min=5, max=80, step=1, value=40.0)
w_yfront = widgets.FloatSlider(description="y_front(m)", min=5, max=160, step=1, value=100.0)

ui = widgets.VBox([
    widgets.HBox([w_speed, w_accel]),
    widgets.HBox([w_tfc, w_trc, w_rearfrac]),
    widgets.HBox([w_dsf, w_dsr]),
    widgets.HBox([w_sm_min, w_sm_max, w_aniso]),
    widgets.HBox([w_pbase, w_dp]),
    widgets.HBox([w_mins, w_penmax]),
    widgets.HBox([w_front_on, w_side_on, w_rear_always]),
    widgets.HBox([w_xlim, w_yback, w_yfront]),
])

out = widgets.interactive_output(
    render_heatmap,
    dict(
        speed_kmh=w_speed,
        accel_mps2=w_accel,
        ego_width_m=widgets.fixed(1.9),

        ttc_front=w_tfc,
        ttc_rear=w_trc,
        d_static=w_dsf,
        d_static_rear=w_dsr,
        rear_frac=w_rearfrac,

        side_margin_min=w_sm_min,
        side_margin_max=w_sm_max,
        anisotropy=w_aniso,

        p_base=w_pbase,
        delta_p=w_dp,

        min_score=w_mins,
        penetration_max_m=w_penmax,

        front_on_kmh=w_front_on,
        side_on_kmh=w_side_on,
        rear_always_on=w_rear_always,

        xlim=w_xlim,
        y_back=w_yback,
        y_front=w_yfront,
    ),
)

display(ui, out)


Output()